In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import pandera as pa
from sqlalchemy import create_engine, inspect, text
from tqdm.auto import tqdm

load_dotenv(find_dotenv(), override=True)

def ingest_csv_or_parquet(
        url: str,
        engine,
        target_table: str,
        source_format: str,
        chunksize: int = 100000
) -> pd.DataFrame:
    
    df_iter = None 
    if source_format == "parquet":
        df = pd.read_parquet(url)
        df.head(0).to_sql(name=target_table, con=engine, if_exists="replace")
        print(f"Table {target_table} created")
        df.to_sql(name=target_table, con=engine, if_exists="append")
        print(f"Inserted the parquet file: {len(df)}")
        return
    else: 
        df_iter = pd.read_csv(url, iterator=True, chunksize=chunksize)

    first_chunk = next(df_iter)
    
    # Infer schema from first chunk
    print(f"Detected columns: {list(first_chunk.columns)}")
    print("Inferring schema from the first chunk...")
    schema = pa.infer_schema(first_chunk)
    
    print(f"Inferred schema:\n{schema}")

    # Create table with schema
    first_chunk.head(0).to_sql(name=target_table, con=engine, if_exists="replace")
    print(f"Table {target_table} created")

    # Validate and insert first chunk
    try:
        validated_chunk = schema.validate(first_chunk, lazy=False)
        validated_chunk.to_sql(name=target_table, con=engine, if_exists="append")
        print(f"Inserted first chunk: {len(first_chunk)}")
    except pa.errors.SchemaError as exc:
        print(f"Schema validation failed on first chunk: {exc}")
        return

    # Process remaining chunks
    for df_chunk in tqdm(df_iter):
        try:
            validated_chunk = schema.validate(df_chunk, lazy=False)
            validated_chunk.to_sql(name=target_table, con=engine, if_exists="append")
            print(f"Inserted chunk: {len(df_chunk)}")
        except pa.errors.SchemaError as exc:
            print(f"Schema validation failed: {exc}")
            return

    print(f"Done ingesting to {target_table}")

pg_user = os.getenv("POSTGRES_USER")
pg_pass = os.getenv("POSTGRES_PASSWORD")
pg_host = "localhost"
pg_port = os.getenv("HOST_PORT")
pg_db = os.getenv("POSTGRES_DB")

chunksize = 100000
engine = create_engine(f"postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}")

In [ ]:
# target_table = "green_tripdata"
# url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-11.parquet" 

# ingest_csv_or_parquet(
#     url=url,
#     engine=engine,
#     target_table=target_table,
#     source_format="parquet",
#     chunksize=chunksize
# )

Table green_tripdata created
Inserted the parquet file: 46912


In [ ]:
target_table = "taxi_zone_lookup"
url = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/misc/taxi_zone_lookup.csv" 

ingest_csv_or_parquet(
    url=url,
    engine=engine,
    target_table=target_table,
    source_format="csv",
    chunksize=chunksize
)

In [7]:
inspector = inspect(engine)
tables = inspector.get_table_names()
print(f"Available tables: {tables}")
print(f"taxi_zone_lookup exists: {'taxi_zone_lookup' in tables}")

Available tables: ['green_tripdata', 'taxi_zone_lookup']
taxi_zone_lookup exists: True


In [12]:
# Method 1: Using SQLAlchemy Inspector (recommended)
inspector = inspect(engine)
columns = inspector.get_columns('green_tripdata')
column_names = [col['name'] for col in columns]
print(f"Column names: {column_names}")

Column names: ['index', 'LocationID', 'Borough', 'Zone', 'service_zone']


In [10]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM taxi_zone_lookup LIMIT 3"))
    for row in result:
        print(row)

(0, 1, 'EWR', 'Newark Airport', 'EWR')
(1, 2, 'Queens', 'Jamaica Bay', 'Boro Zone')
(2, 3, 'Bronx', 'Allerton/Pelham Gardens', 'Boro Zone')


In [11]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT * FROM green_tripdata LIMIT 3"))
    for row in result:
        print(row)

(0, 2, datetime.datetime(2025, 11, 1, 0, 34, 48), datetime.datetime(2025, 11, 1, 0, 41, 39), 'N', 1.0, 74, 42, 1.0, 0.74, 7.2, 1.0, 0.5, 1.94, 0.0, None, 1.0, 11.64, 1.0, 1.0, 0.0, 0.0)
(1, 2, datetime.datetime(2025, 11, 1, 0, 18, 52), datetime.datetime(2025, 11, 1, 0, 24, 27), 'N', 1.0, 74, 42, 2.0, 0.95, 7.2, 1.0, 0.5, 0.0, 0.0, None, 1.0, 9.7, 2.0, 1.0, 0.0, 0.0)
(2, 2, datetime.datetime(2025, 11, 1, 1, 3, 14), datetime.datetime(2025, 11, 1, 1, 15, 24), 'N', 1.0, 83, 160, 1.0, 2.19, 13.5, 1.0, 0.5, 5.0, 0.0, None, 1.0, 21.0, 1.0, 1.0, 0.0, 0.0)
